This file will be used to store common functions used by the Snowflake ingestion framework. No non-standard libraries should be imported as it can cause issues for other people as their notebooks will fail unless they pip install the non-standard library.

In [0]:
# Snowflake secret scope name - change this variable to point to your scope
SNOWFLAKE_SECRET_SCOPE = "snowflake_credentials"

In [0]:
def get_snowflake_data(database, schema, table_name):
    """
    Returns a dataframe with the data from a Snowflake table using the
    Databricks Snowflake connector with secrets-based authentication.

    Args:
        database (str): Snowflake database name
        schema (str): Snowflake schema name
        table_name (str): Snowflake table name

    Returns:
        DataFrame: A PySpark dataframe with the source data
    """

    sf_options = {
        "sfUrl": dbutils.secrets.get(SNOWFLAKE_SECRET_SCOPE, "host"),
        "sfUser": dbutils.secrets.get(SNOWFLAKE_SECRET_SCOPE, "user"),
        "sfPassword": dbutils.secrets.get(SNOWFLAKE_SECRET_SCOPE, "password"),
        "sfDatabase": database,
        "sfSchema": schema,
        "sfWarehouse": dbutils.secrets.get(SNOWFLAKE_SECRET_SCOPE, "warehouse"),
        "sfRole": dbutils.secrets.get(SNOWFLAKE_SECRET_SCOPE, "role"),
    }

    df = (spark.read
        .format("snowflake")
        .options(**sf_options)
        .option("dbtable", table_name)
        .load()
    )

    return df

In [0]:
def dynamic_merge_sql(table_name, match_columns, update_columns, insert_columns):
    """
    Build dynamic merge statement for use when merging with any code.

    Args:
        table_name (str): The name of the Delta Lake table.
        match_columns (list): A list of the columns to match on aka the tables primary/composite key
        update_columns (list): Columns which will be updated when a record matches
        insert_columns (list): All columns in the table for the insert part where records don't match

    Returns:
        str: A string with the merge statement for this table
    """

    # Create the "USING" clause for matching columns
    using_clause = "USING updates AS updates ON "
    using_clause += " AND ".join([f"target.{col} = updates.{col}" for col in match_columns])

    # Create the "WHEN MATCHED" clause for updating columns
    update_clause = "WHEN MATCHED THEN UPDATE SET "
    update_clause += ", ".join([f"target.{col} = updates.{col}" for col in update_columns])

    # Create the "WHEN NOT MATCHED" clause for inserting all columns
    insert_clause = "WHEN NOT MATCHED THEN INSERT ("
    insert_clause += ", ".join(insert_columns) + ") VALUES ("
    insert_clause += ", ".join([f"updates.{col}" for col in insert_columns]) + ")"

    # Assemble the complete MERGE statement
    merge_sql = f"""
    MERGE INTO {table_name} AS target
    {using_clause}
    {update_clause}
    {insert_clause}
    """

    return merge_sql

In [0]:
def vacuum_table(table_name, retain_hours):
    """
    Vacuum a Databricks Delta Lake table to remove old data files.

    Args:
        table_name (str): The name of the Delta Lake table.
        retain_hours (int): The number of hours to retain data files.

    Returns:
        str: A message indicating the result of the VACUUM operation.
    """
    try:
        vacuum_sql = f"VACUUM {table_name} RETAIN {retain_hours} HOURS"
        spark.sql(vacuum_sql)
        return f"VACUUM operation on table '{table_name}' completed successfully."
    except Exception as e:
        return f"Error performing VACUUM operation on table '{table_name}': {str(e)}"

In [0]:
from pyspark.sql import DataFrame

def generate_match_insert_columns(match_columns, dataframe):
    """
    Get a list of column names from a PySpark DataFrame.

    Args:
        match_columns (str): Comma-separated list of the columns to match on
        dataframe (DataFrame): The PySpark DataFrame.

    Returns:
        tuple: (match_columns_list, update_list, all_columns)
    """

    match_columns_list = match_columns.split(", ")
    all_columns = dataframe.columns
    update_list = [x for x in all_columns if x not in match_columns_list]

    return match_columns_list, update_list, all_columns

In [0]:
def validate_cluster_by(cluster_by):
    """
    Validate the cluster_by configuration. Must be a list of 1-4 column names
    or ["auto"]. Raises ValueError if invalid.

    Args:
        cluster_by (list): List of column names or ["auto"]

    Returns:
        list: The validated cluster_by list
    """
    if not isinstance(cluster_by, list) or len(cluster_by) == 0:
        raise ValueError("cluster_by must be a non-empty list.")
    
    if len(cluster_by) > 4:
        raise ValueError(
            f"cluster_by supports a maximum of 4 columns, but {len(cluster_by)} were provided: {cluster_by}"
        )

    return cluster_by